In [9]:
import os
import json
import pickle
import joblib
import csv
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

# Folder to store all the scalers and models
ARTIFACT_DIR = "artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

In [10]:
# LOAD DATA
print("Loading JSON files...")
with open('fulldb-entities.json', 'r') as db_json:
    db = json.load(db_json)
with open('processed_labels/removal_labels.json', 'r') as labels_json:
    labels = json.load(labels_json)
with open('processed_labels/test_sample.json', 'r') as test_file:
    test = json.load(test_file)

dataset_cuis = sorted([cui for cui in labels.keys() if cui in db and cui not in test])
print(f"Unique Entities in train set = {len(dataset_cuis)}")

test_cuis = sorted([cui for cui in test.keys() if cui in db])
print(f"Unique Entities in test set  = {len(test_cuis)}")

Loading JSON files...
Unique Entities in train set = 1055
Unique Entities in test set  = 76


In [11]:
# PATCH SEMANTIC TYPES FROM CSV
print("\nPatching Semantic Types from combined.csv...")
# We read the CSV to grab the semantic types since they were missing in Neo4j
try:
    df_combined = pd.read_csv("annotated/combined.csv", sep=None, engine='python', encoding='utf-8-sig')
    
    # Quick sanity check to see what columns Pandas found
    if 'CUI' not in df_combined.columns:
        print(f"ERROR: 'CUI' column not found. Found these instead: {df_combined.columns.tolist()}")
    else:
        for index, row in df_combined.iterrows():
            cui = str(row['CUI']).strip()
            
            if cui in db:
                raw_sty = str(row['Semantic Types'])
                
                # Clean the string (e.g., turn '["neop"]' into 'neop')
                clean_sty = raw_sty.replace('[', '').replace(']', '').replace('"', '').replace("'", "")
                sty_list = [s.strip() for s in clean_sty.split(',') if s.strip()]
                
                db[cui]["sem_types"] = sty_list
                
        print("Patch complete!")

except Exception as e:
    print(f"Could not read combined.csv: {e}")


Patching Semantic Types from combined.csv...
Patch complete!


In [12]:
# SEMANTIC TYPES & L1 FEATURE SELECTION
print("\nExtracting and selecting Semantic Types...")
sem_types = []
for cui in db.keys():
    for sem_type in db[cui]["sem_types"]:
        if sem_type not in sem_types:
            sem_types.append(sem_type)

sem_types_one_hot = []
for cui in dataset_cuis:
    row = [1 if st in db[cui]["sem_types"] else 0 for st in sem_types]
    sem_types_one_hot.append(row)
X_sem = np.asarray(sem_types_one_hot)

Y = np.asarray([labels[cui] for cui in dataset_cuis])

# Use L1 to find important semantic types
coefs = LogisticRegression(penalty='l1', solver='liblinear', random_state=2020).fit(X_sem, Y).coef_[-1]
sem_type_imporances = sorted(list(zip(sem_types, coefs)), key=lambda x: abs(x[1]), reverse=True)

selected_sem_types = []
for sem_type, coef in sem_type_imporances:
    if abs(coef) > 0:
        selected_sem_types.append(sem_type)

print(f"Reduced from {len(sem_types)} to {len(selected_sem_types)} important Semantic Types.")
pickle.dump(selected_sem_types, open(f"{ARTIFACT_DIR}/selected_sem_types.pkl", "wb"))
joblib.dump(selected_sem_types, f"{ARTIFACT_DIR}/selected_sem_types.joblib")

selected_sem_types_one_hot = []
for cui in dataset_cuis:
    row = [1 if st in db[cui]["sem_types"] else 0 for st in selected_sem_types]
    selected_sem_types_one_hot.append(row)
X_selected_sem = np.asarray(selected_sem_types_one_hot)


Extracting and selecting Semantic Types...
Reduced from 125 to 53 important Semantic Types.


In [13]:
# PREPARE CENTRALITY & EMBEDDINGS
print("\nScaling Centrality & Graph Embeddings...")

# PageRank
pagerank = {cui: db[cui].get("pagerank", 0) for cui in dataset_cuis}
X_pagerank = np.asarray([[pagerank[cui] if pagerank[cui] else 0] for cui in dataset_cuis])
pagerank_scaler = MinMaxScaler()
X_pagerank = pagerank_scaler.fit_transform(X_pagerank)
joblib.dump(pagerank_scaler, f"{ARTIFACT_DIR}/pagerank_scaler.joblib")

# Embeddings (The Tournament)
algos = ["node2vec", "fastrp", "graphsage", "hashgnn"]
dims = ["32", "64", "128"]

X_embeddings = {}
embedding_scalers = {}

for algo in algos:
    for dim in dims:
        key = f"{algo}{dim}"
        # Fetch the list of numbers from JSON. If missing, create a zero array.
        X_embeddings[key] = np.asarray([db[cui].get(key, [0]*int(dim)) for cui in dataset_cuis])
        
        # Scale
        scaler = MinMaxScaler()
        X_embeddings[key] = scaler.fit_transform(X_embeddings[key])
        joblib.dump(scaler, f"{ARTIFACT_DIR}/{key}_scaler.joblib")


Scaling Centrality & Graph Embeddings...


In [14]:

# BUILD COMBINATIONS
feature_combinations = {}
# Add baseline Semantic Types
feature_combinations["selected_sem_types"] = X_selected_sem

for algo in algos:
    for dim in dims:
        base = f"{algo}{dim}"
        
        # 1. Just the Embedding
        feature_combinations[f"{base}"] = X_embeddings[base]
        # 2. Embedding + PageRank
        feature_combinations[f"{base} + Pagerank"] = np.concatenate([X_embeddings[base], X_pagerank], axis=1)
        # 3. Embedding + Selected Semantics
        feature_combinations[f"{base} + Sem"] = np.concatenate([X_embeddings[base], X_selected_sem], axis=1)
        # 4. Embedding + Selected Semantics + PageRank
        feature_combinations[f"{base} + Sem + Pagerank"] = np.concatenate([X_embeddings[base], X_selected_sem, X_pagerank], axis=1)

In [15]:
# RUN CROSS-VALIDATION TOURNAMENT
print("\nRunning 10-Fold CV Tournament... ")
feature_scores = {}
scoring = 'f1_micro'

print(f"{'Features':<35} | {'Dims':<4} | {'F1 Score':<15}")
print("-" * 60)

for name, X in feature_combinations.items():
    cv_results = cross_validate(
        LogisticRegression(penalty='l2', solver='liblinear', random_state=2020),
        X, Y, cv=10, scoring=scoring
    )
    median_f1 = np.median(cv_results['test_score'])
    feature_scores[name] = median_f1
    print(f"{name:<35} | {X.shape[1]:<4} | {median_f1:.3f} (+/-{np.std(cv_results['test_score']):.3f})")

# Find the absolute best
best_combo_name = max(feature_scores, key=feature_scores.get)
print("-" * 60)
print(f"WINNER: {best_combo_name} with F1 = {feature_scores[best_combo_name]:.3f}")


Running 10-Fold CV Tournament... 
Features                            | Dims | F1 Score       
------------------------------------------------------------
selected_sem_types                  | 53   | 0.654 (+/-0.095)
node2vec32                          | 32   | 0.662 (+/-0.080)
node2vec32 + Pagerank               | 33   | 0.662 (+/-0.080)
node2vec32 + Sem                    | 85   | 0.676 (+/-0.084)
node2vec32 + Sem + Pagerank         | 86   | 0.676 (+/-0.084)
node2vec64                          | 64   | 0.662 (+/-0.078)
node2vec64 + Pagerank               | 65   | 0.662 (+/-0.078)
node2vec64 + Sem                    | 117  | 0.676 (+/-0.085)
node2vec64 + Sem + Pagerank         | 118  | 0.676 (+/-0.085)
node2vec128                         | 128  | 0.667 (+/-0.078)
node2vec128 + Pagerank              | 129  | 0.667 (+/-0.078)
node2vec128 + Sem                   | 181  | 0.686 (+/-0.080)
node2vec128 + Sem + Pagerank        | 182  | 0.686 (+/-0.080)
fastrp32                            |

warning! den exw asxolhthei me tis parametrous gia ta parapanw runs + vgainoun polla nodes xwris relationships, [Pagerank 0] (isws kwn/nou)

In [16]:
# EXPORT BEST TO CSV FOR TRAINING
print("\nExporting train_df.csv and test_df.csv using the winning combination...")

# Get the best matrix
X_best = feature_combinations[best_combo_name]
num_features = X_best.shape[1]
feature_names = [f"feat_{i}" for i in range(num_features)]
column_names = ['label'] + feature_names + ['target']

# Train DataFrame
data_train = []
for ii, cui in enumerate(dataset_cuis):
    data_train.append([db[cui]['label']] + X_best[ii, :].tolist() + [Y[ii]])
train_df = pd.DataFrame(data=data_train, columns=column_names, index=dataset_cuis)
train_df.to_csv("./train_df.csv")

# Test DataFrame prep (we need to apply the exact same transformation to the test set)
# To keep this script concise and robust, we export all test features matching the winning dimensions.
# Note: For strict thesis recreation, we will apply the winning scalers below.

# # --- Prepare Test Set based on Winner --- (old version, commented out)
# X_test_parts = []
# if "Sem" in best_combo_name or best_combo_name == "selected_sem_types":
#     # Selected Sem Types for Test
#     test_sem_hot = []
#     for cui in test_cuis:
#         row = [1 if st in db[cui]["sem_types"] else 0 for st in selected_sem_types]
#         test_sem_hot.append(row)
#     X_test_parts.append(np.asarray(test_sem_hot))

# if "Pagerank" in best_combo_name:
#     # PageRank for Test
#     test_pagerank = np.asarray([[db[cui].get("pagerank", 0) if db[cui].get("pagerank") else 0] for cui in test_cuis])
#     X_test_parts.append(pagerank_scaler.transform(test_pagerank))

# # Extract the algorithm base name (e.g., node2vec32) if it's in the winner name
# algo_base = best_combo_name.split(" + ")[0]
# if algo_base in X_embeddings:
#     test_emb = np.asarray([db[cui].get(algo_base, [0]*int(algo_base.replace(algo_base.rstrip('0123456789'), ''))) for cui in test_cuis])
#     emb_scaler = joblib.load(f"{ARTIFACT_DIR}/{algo_base}_scaler.joblib")
#     X_test_parts.append(emb_scaler.transform(test_emb))

# if len(X_test_parts) > 1:
#     X_test_final = np.concatenate(X_test_parts, axis=1)
# elif len(X_test_parts) == 1:
#     X_test_final = X_test_parts[0]
# else:
#     # Fallback if something weird happened
#     X_test_final = X_test_parts[0]

# data_test = []
# for ii, cui in enumerate(test_cuis):
#     data_test.append([db[cui]['label']] + X_test_final[ii, :].tolist() + [test[cui]])
    
# test_df = pd.DataFrame(data=data_test, columns=column_names, index=test_cuis)
# test_df.to_csv("./test_df.csv")


# --- Prepare Test Set based on Winner ---
X_test_parts = []
algo_base = best_combo_name.split(" + ")[0]

# 1. EMBEDDINGS FIRST (To match Train)
if algo_base in X_embeddings:
    test_emb = np.asarray([db[cui].get(algo_base, [0]*int(algo_base.replace(algo_base.rstrip('0123456789'), ''))) for cui in test_cuis])
    emb_scaler = joblib.load(f"{ARTIFACT_DIR}/{algo_base}_scaler.joblib")
    X_test_parts.append(emb_scaler.transform(test_emb))

# 2. SEMANTIC TYPES SECOND (To match Train)
if "Sem" in best_combo_name or best_combo_name == "selected_sem_types":
    test_sem_hot = []
    for cui in test_cuis:
        row = [1 if st in db[cui].get("sem_types", []) else 0 for st in selected_sem_types]
        test_sem_hot.append(row)
    X_test_parts.append(np.asarray(test_sem_hot))

# 3. PAGERANK THIRD (To match Train)
if "Pagerank" in best_combo_name:
    test_pagerank = np.asarray([[db[cui].get("pagerank", 0) if db[cui].get("pagerank") else 0] for cui in test_cuis])
    X_test_parts.append(pagerank_scaler.transform(test_pagerank))

# Safely Concatenate
if len(X_test_parts) > 1:
    X_test_final = np.concatenate(X_test_parts, axis=1)
elif len(X_test_parts) == 1:
    X_test_final = X_test_parts[0]

data_test = []
for ii, cui in enumerate(test_cuis):
    data_test.append([db[cui]['label']] + X_test_final[ii, :].tolist() + [test[cui]])
    
test_df = pd.DataFrame(data=data_test, columns=column_names, index=test_cuis)
test_df.to_csv("./test_df.csv")

print("train_df.csv and test_df.csv are ready for the ML models.")


Exporting train_df.csv and test_df.csv using the winning combination...
train_df.csv and test_df.csv are ready for the ML models.
